In [1]:
# ─── Celda 01: Imports y configuración ───────────────────────────────────────
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Rutas ──
BASE        = Path("/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks")
SPATIAL_DIR = BASE / "data/spatial_outputs"
MANIFEST    = SPATIAL_DIR / "manifest.csv"
BOXES_DIR   = SPATIAL_DIR / "boxes"
MASKS_DIR   = SPATIAL_DIR / "masks"
FEAT_DIR    = SPATIAL_DIR / "features"
CKPT_DIR    = SPATIAL_DIR / "checkpoints_03"
OUT_DIR     = SPATIAL_DIR / "spatiotemporal_outputs"

for d in [CKPT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hardware ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo : {DEVICE}")
if DEVICE.type == "cuda":
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM libre  : {free/1e9:.2f} GB / {total/1e9:.2f} GB")
print("✓ Imports completos")

/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/env_ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo : cuda
GPU         : NVIDIA GeForce GTX 1660 Ti with Max-Q Design
VRAM libre  : 6.09 GB / 6.23 GB
✓ Imports completos


In [2]:
# ─── Celda 02: Cargar manifest y analizar secuencias ─────────────────────────
df = pd.read_csv(MANIFEST)
print(f"Total imágenes : {len(df):,}")
print(f"Columnas       : {list(df.columns)}\n")

# Construcción de seq_id — primeros 8 caracteres del image_id
N_CHARS = 8
df["seq_id"] = df["image_id"].str[:N_CHARS]

# ── Distribución de splits ──
print("── Split ──")
print(df["split"].value_counts().to_string(), "\n")

# ── Análisis de secuencias por split ──
print("── Secuencias por split ──")
for split in ["train", "val", "test"]:
    sub      = df[df["split"] == split]
    seq_lens = sub.groupby("seq_id").size()
    viables  = (seq_lens >= 10).sum()
    print(f"  {split}: {len(seq_lens):,} secuencias | "
          f"{viables:,} con ≥10 frames ({viables/len(seq_lens)*100:.1f}%)")

# ── Distribución global de longitudes ──
all_lens = df.groupby("seq_id").size()
print(f"\nLongitud de secuencia:")
print(f"  mediana : {all_lens.median():.0f}")
print(f"  p10     : {all_lens.quantile(0.1):.0f}")
print(f"  p90     : {all_lens.quantile(0.9):.0f}")
print(f"  max     : {all_lens.max()}")
print(f"  total secuencias : {len(all_lens):,}")

Total imágenes : 61,345
Columnas       : ['image_id', 'split', 'weather', 'timeofday', 'clahe_mode', 'num_objects', 'path_curada', 'path_boxes', 'path_mask', 'path_features']

── Split ──
split
train    42941
val       9202
test      9202 

── Secuencias por split ──
  train: 29,419 secuencias | 39 con ≥10 frames (0.1%)
  val: 8,280 secuencias | 0 con ≥10 frames (0.0%)
  test: 8,243 secuencias | 0 con ≥10 frames (0.0%)

Longitud de secuencia:
  mediana : 1
  p10     : 1
  p90     : 3
  max     : 41
  total secuencias : 37,784


In [3]:
# ─── Celda 02b: Construir pseudo-secuencias por weather+timeofday ─────────────
import hashlib

# Clave de agrupación: condiciones visuales similares dentro del mismo split
df["seq_id"] = (df["split"] + "_" +
                df["weather"].fillna("unknown") + "_" +
                df["timeofday"].fillna("unknown"))

# Orden determinista dentro de cada grupo: por num_objects (proxy de densidad)
# Simula progresión de escenas simples → complejas dentro de la misma condición
df = df.sort_values(["seq_id", "num_objects"]).reset_index(drop=True)

# ── Análisis post-agrupación ──
print("── Pseudo-secuencias por split ──")
T_VENTANA = 10
STEP      = 5

total_ventanas = 0
for split in ["train", "val", "test"]:
    sub      = df[df["split"] == split]
    seq_lens = sub.groupby("seq_id").size()
    viables  = (seq_lens >= T_VENTANA).sum()
    ventanas = seq_lens[seq_lens >= T_VENTANA].apply(
        lambda n: max(1, (n - T_VENTANA) // STEP + 1)
    ).sum()
    total_ventanas += ventanas
    print(f"  {split}: {len(seq_lens)} grupos | "
          f"{viables} con ≥{T_VENTANA} frames | "
          f"~{ventanas:,} ventanas")

print(f"\nTotal ventanas de entrenamiento : {total_ventanas:,}")
print(f"T={T_VENTANA} frames | step={STEP} (overlap 50%)")
print(f"\nEjemplos de seq_id:")
print(df["seq_id"].unique()[:8])

── Pseudo-secuencias por split ──
  train: 25 grupos | 21 con ≥10 frames | ~8,556 ventanas
  val: 24 grupos | 18 con ≥10 frames | ~1,811 ventanas
  test: 24 grupos | 17 con ≥10 frames | ~1,808 ventanas

Total ventanas de entrenamiento : 12,175
T=10 frames | step=5 (overlap 50%)

Ejemplos de seq_id:
['test_clear_dawn/dusk' 'test_clear_daytime' 'test_clear_night'
 'test_foggy_dawn/dusk' 'test_foggy_daytime' 'test_foggy_night'
 'test_overcast_dawn/dusk' 'test_overcast_daytime']


In [5]:
# ─── Celda 03: Construcción del grafo multiescala por frame ──────────────────
CLASES_YOLO  = ["person","rider","car","truck","bus",
                 "train","motorcycle","bicycle","traffic light"]
SIGMA_LOCAL  = 100.0
SIGMA_GLOBAL = 300.0
THRESH_LOCAL = 150.0
THRESH_GLOBAL= 400.0
NODE_DIM     = 4 + 9 + 1 + 1 + 2048   # = 2063

def centroide(bbox):
    x1, y1, x2, y2 = bbox
    return (x1 + x2) / 2, (y1 + y2) / 2

def mask_zone(mask_arr, bbox):
    x1, y1, x2, y2 = (int(v) for v in bbox)
    region = mask_arr[max(0,y1):y2, max(0,x1):x2]
    if region.size == 0:
        return 0.0
    vals, counts = np.unique(region[region != 255], return_counts=True)
    return float(vals[counts.argmax()]) / 18.0 if len(vals) else 0.0

def edges_por_distancia(centroides, thresh, sigma):
    N = len(centroides)
    src, dst, weights = [], [], []
    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            dx = centroides[i][0] - centroides[j][0]
            dy = centroides[i][1] - centroides[j][1]
            d  = (dx**2 + dy**2) ** 0.5
            if d <= thresh:
                src.append(i)
                dst.append(j)
                weights.append(float(np.exp(-(d**2) / (sigma**2))))
    if not src:
        src = dst = list(range(N))
        weights = [1.0] * N
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr  = torch.tensor(weights, dtype=torch.float).unsqueeze(1)
    return edge_index, edge_attr

def build_graph(boxes_path, mask_path, feat_path):
    try:
        with open(boxes_path) as f:
            boxes = json.load(f).get("boxes", [])
    except Exception:
        boxes = []

    if len(boxes) == 0:
        return None

    mask = (np.load(mask_path)
            if Path(mask_path).exists()
            else np.zeros((320, 320), dtype=np.uint8))

    feat_g = (torch.load(feat_path, map_location="cpu",
                         weights_only=True).float()
              if Path(feat_path).exists()
              else torch.zeros(2048))

    node_feats, centroides = [], []
    for b in boxes:
        cls_name = b.get("class_name", "")
        cls_idx  = CLASES_YOLO.index(cls_name) if cls_name in CLASES_YOLO else 0
        onehot   = [0.0] * 9
        onehot[cls_idx] = 1.0
        bbox     = b["bbox_xyxy"]
        cx, cy   = centroide(bbox)
        centroides.append((cx, cy))
        mz       = mask_zone(mask, bbox)
        nf       = bbox + onehot + [float(b.get("confidence", 0.0)), mz] + feat_g.tolist()
        node_feats.append(nf)

    x = torch.tensor(node_feats, dtype=torch.float)
    ei_local,  ea_local  = edges_por_distancia(centroides, THRESH_LOCAL,  SIGMA_LOCAL)
    ei_global, ea_global = edges_por_distancia(centroides, THRESH_GLOBAL, SIGMA_GLOBAL)

    return Data(x=x,
                edge_index=ei_local,    edge_attr=ea_local,
                edge_index_g=ei_global, edge_attr_g=ea_global)

# ── Prueba con indicador de progreso ──
print("Buscando frame con detecciones...", end=" ")
filas_con_boxes = df[df["path_boxes"].apply(lambda p: Path(p).exists())]
print(f"{len(filas_con_boxes):,} frames con archivo boxes encontrados")

print("Cargando boxes...", end=" ")
row_test = filas_con_boxes.iloc[0]
print(f"OK → {row_test.image_id}")

print("Construyendo grafo...", end=" ")
g = build_graph(row_test.path_boxes, row_test.path_mask, row_test.path_features)

if g:
    print("OK\n")
    print(f"✓ Grafo construido")
    print(f"  Nodos          : {g.num_nodes}")
    print(f"  Edges locales  : {g.edge_index.shape[1]}")
    print(f"  Edges globales : {g.edge_index_g.shape[1]}")
    print(f"  Feature dim    : {g.x.shape[1]}")
else:
    print("sin objetos — probando siguiente frame...")
    row_test = filas_con_boxes.iloc[10]
    g = build_graph(row_test.path_boxes, row_test.path_mask, row_test.path_features)
    if g:
        print(f"✓ Grafo construido en frame alternativo")
        print(f"  Nodos          : {g.num_nodes}")
        print(f"  Feature dim    : {g.x.shape[1]}")
    else:
        print("✗ Revisar archivos boxes — YOLO no detectó objetos en muestra")

Buscando frame con detecciones... 61,345 frames con archivo boxes encontrados
Cargando boxes... OK → 063eb66e-f58f45b7.jpg
Construyendo grafo... OK

✓ Grafo construido
  Nodos          : 5
  Edges locales  : 6
  Edges globales : 20
  Feature dim    : 2063


In [6]:
# ─── Celda 04: Señal de riesgo proxy ─────────────────────────────────────────
CRITICOS  = {"person", "rider", "car", "truck", "bus", "motorcycle"}
FPS_PROXY = 15.0

def ttc_score(boxes_curr, boxes_prev):
    if not boxes_prev or not boxes_curr:
        return 0.0
    min_ttc = float("inf")
    for bc in boxes_curr:
        if bc.get("class_name") not in CRITICOS:
            continue
        cx1, cy1 = centroide(bc["bbox_xyxy"])
        for bp in boxes_prev:
            if bp.get("class_name") not in CRITICOS:
                continue
            cx2, cy2 = centroide(bp["bbox_xyxy"])
            d = max(((cx1-cx2)**2 + (cy1-cy2)**2)**0.5, 1e-6)
            v = d * FPS_PROXY
            min_ttc = min(min_ttc, d / v)
    if min_ttc == float("inf"):
        return 0.0
    return float(np.clip(1.0 - min_ttc / 5.0, 0.0, 1.0))

def density_score(boxes):
    return float(np.clip(len(boxes) / 15.0, 0.0, 1.0))

def proximity_score(boxes, umbral_px=80.0):
    criticos = [b for b in boxes if b.get("class_name") in CRITICOS]
    if len(criticos) < 2:
        return 0.0
    min_d = float("inf")
    for i in range(len(criticos)):
        for j in range(i + 1, len(criticos)):
            cx1, cy1 = centroide(criticos[i]["bbox_xyxy"])
            cx2, cy2 = centroide(criticos[j]["bbox_xyxy"])
            min_d = min(min_d, ((cx1-cx2)**2 + (cy1-cy2)**2)**0.5)
    return float(np.clip(1.0 - min_d / umbral_px, 0.0, 1.0))

def risk_label(boxes_curr, boxes_prev=None):
    t = ttc_score(boxes_curr, boxes_prev)
    d = density_score(boxes_curr)
    p = proximity_score(boxes_curr)
    return round(0.5*t + 0.2*d + 0.3*p, 4)

# ── Prueba ──
print("Probando señal de riesgo...", end=" ")
with open(row_test.path_boxes) as f:
    boxes_ex = json.load(f)["boxes"]

t = ttc_score(boxes_ex, None)
d = density_score(boxes_ex)
p = proximity_score(boxes_ex)
r = risk_label(boxes_ex)

print("OK\n")
print(f"  Objetos detectados : {len(boxes_ex)}")
print(f"  ttc_score          : {t:.4f}  (0 porque no hay frame previo)")
print(f"  density_score      : {d:.4f}  ({len(boxes_ex)}/15 objetos)")
print(f"  proximity_score    : {p:.4f}")
print(f"  risk_label         : {r:.4f}")

Probando señal de riesgo... OK

  Objetos detectados : 5
  ttc_score          : 0.0000  (0 porque no hay frame previo)
  density_score      : 0.3333  (5/15 objetos)
  proximity_score    : 0.0000
  risk_label         : 0.0667


In [7]:
# ─── Celda 05: Dataset con sliding window por pseudo-secuencia ────────────────
class SpatiotemporalDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_df, split, t_ventana=10, step=5):
        self.t  = t_ventana
        sub     = manifest_df[manifest_df["split"] == split].copy()
        self.windows = []

        grupos = sub.groupby("seq_id")
        for seq_id, grupo in tqdm(grupos, desc=f"Indexando {split}", leave=False):
            grupo = grupo.reset_index(drop=True)
            if len(grupo) < t_ventana:
                continue
            for start in range(0, len(grupo) - t_ventana + 1, step):
                self.windows.append(grupo.iloc[start:start + t_ventana].copy())

        print(f"  Split '{split}': {len(grupos):,} grupos → {len(self.windows):,} ventanas")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        window     = self.windows[idx]
        graphs     = []
        risks      = []
        prev_boxes = None

        for _, row in window.iterrows():
            # Cargar boxes
            try:
                with open(row.path_boxes) as f:
                    boxes = json.load(f)["boxes"]
            except Exception:
                boxes = []

            # Construir grafo — si no hay objetos usar grafo vacío
            g = build_graph(row.path_boxes, row.path_mask, row.path_features)
            if g is None:
                g = Data(
                    x            = torch.zeros((1, NODE_DIM)),
                    edge_index   = torch.zeros((2, 1), dtype=torch.long),
                    edge_attr    = torch.zeros((1, 1)),
                    edge_index_g = torch.zeros((2, 1), dtype=torch.long),
                    edge_attr_g  = torch.zeros((1, 1))
                )

            graphs.append(g)
            risks.append(risk_label(boxes, prev_boxes))
            prev_boxes = boxes

        risk_score = torch.tensor([float(np.mean(risks))], dtype=torch.float)
        return graphs, risk_score

# ── Construir datasets ──
print("Construyendo datasets...")
ds_train = SpatiotemporalDataset(df, "train")
ds_val   = SpatiotemporalDataset(df, "val")
ds_test  = SpatiotemporalDataset(df, "test")

# ── Verificar un item ──
print("\nVerificando item de prueba...", end=" ")
grafos, risk = ds_train[0]
print("OK")
print(f"  Grafos en ventana : {len(grafos)}")
print(f"  Nodos grafo[0]    : {grafos[0].num_nodes}")
print(f"  Risk score        : {risk.item():.4f}")

Construyendo datasets...


  Split 'train': 25 grupos → 8,556 ventanas


  Split 'val': 24 grupos → 1,811 ventanas


  Split 'test': 24 grupos → 1,808 ventanas

Verificando item de prueba... OK
  Grafos en ventana : 10
  Nodos grafo[0]    : 1
  Risk score        : 0.1482


In [8]:
# ─── Celda 06: DataLoader con collate personalizado ───────────────────────────
def collate_sequences(batch):
    """
    batch: lista de (graphs_list_T, risk_score)
    Retorna: (lista de T Batch PyG, risk_tensor [B,1])
    """
    all_graphs = [item[0] for item in batch]   # [B][T]
    risks      = torch.stack([item[1] for item in batch])   # (B, 1)
    T_len      = len(all_graphs[0])

    batched_t = []
    for t in range(T_len):
        grafos_t = [seq[t] for seq in all_graphs]
        batched_t.append(Batch.from_data_list(grafos_t))

    return batched_t, risks

BATCH = 4

dl_train = torch.utils.data.DataLoader(
    ds_train, batch_size=BATCH, shuffle=True,
    num_workers=2, collate_fn=collate_sequences,
    pin_memory=True)

dl_val = torch.utils.data.DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    num_workers=2, collate_fn=collate_sequences)

dl_test = torch.utils.data.DataLoader(
    ds_test, batch_size=BATCH, shuffle=False,
    num_workers=2, collate_fn=collate_sequences)

# ── Verificar un batch ──
print("Verificando batch...", end=" ")
batch_grafos, batch_risks = next(iter(dl_train))
print("OK\n")
print(f"  Timesteps por batch : {len(batch_grafos)}")
print(f"  Nodos en t=0        : {batch_grafos[0].num_nodes}")
print(f"  Risk shape          : {batch_risks.shape}")
print(f"  Risk valores        : {batch_risks.squeeze().tolist()}")
print(f"\n  Train batches : {len(dl_train)}")
print(f"  Val batches   : {len(dl_val)}")

Verificando batch... OK

  Timesteps por batch : 10
  Nodos en t=0        : 21
  Risk shape          : torch.Size([4, 1])
  Risk valores        : [0.4969800114631653, 0.4477800130844116, 0.826479971408844, 0.8015000224113464]

  Train batches : 2139
  Val batches   : 453


In [9]:
# ─── Celda 07: GCN Encoder multiescala ───────────────────────────────────────
GCN_HIDDEN = 256
GCN_OUT    = 128   # embedding por grafo por escala → concat → 256 → FC → 128

class GCNEncoder(nn.Module):
    def __init__(self, in_dim=NODE_DIM, hidden=GCN_HIDDEN, out_dim=GCN_OUT, dropout=0.3):
        super().__init__()
        # Rama local
        self.conv_l1 = GCNConv(in_dim, hidden)
        self.conv_l2 = GCNConv(hidden, out_dim)
        # Rama global
        self.conv_g1 = GCNConv(in_dim, hidden)
        self.conv_g2 = GCNConv(hidden, out_dim)
        # Fusión de las dos ramas
        self.fc_fuse = nn.Linear(out_dim * 2, out_dim)
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x, edge_index_l, edge_index_g, batch):
        # Rama local
        xl = self.relu(self.conv_l1(x, edge_index_l))
        xl = self.drop(xl)
        xl = self.conv_l2(xl, edge_index_l)
        xl = global_mean_pool(xl, batch)   # (B, GCN_OUT)

        # Rama global
        xg = self.relu(self.conv_g1(x, edge_index_g))
        xg = self.drop(xg)
        xg = self.conv_g2(xg, edge_index_g)
        xg = global_mean_pool(xg, batch)   # (B, GCN_OUT)

        # Concat + fusión
        out = torch.cat([xl, xg], dim=1)   # (B, GCN_OUT*2)
        out = self.relu(self.fc_fuse(out)) # (B, GCN_OUT)
        return out

# ── Verificar con un batch real ──
print("Verificando GCN...", end=" ")
gcn = GCNEncoder().to(DEVICE)

bt = batch_grafos[0].to(DEVICE)
with torch.no_grad():
    emb = gcn(bt.x, bt.edge_index, bt.edge_index_g, bt.batch)

print("OK\n")
print(f"  Parámetros GCN : {sum(p.numel() for p in gcn.parameters()):,}")
print(f"  Input shape    : {bt.x.shape}")
print(f"  Output shape   : {emb.shape}  ← debe ser (4, {GCN_OUT})")

Verificando GCN... OK

  Parámetros GCN : 1,155,456
  Input shape    : torch.Size([21, 2063])
  Output shape   : torch.Size([4, 128])  ← debe ser (4, 128)


In [10]:
# ─── Celda 08: LSTM de memoria temporal ──────────────────────────────────────
LSTM_HIDDEN = 256
LSTM_LAYERS = 2

class RiskLSTM(nn.Module):
    def __init__(self, input_size=GCN_OUT, hidden=LSTM_HIDDEN,
                 layers=LSTM_LAYERS, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.fc  = nn.Linear(hidden, 1)
        self.sig = nn.Sigmoid()

    def forward(self, seq):
        """
        seq : (B, T, GCN_OUT)
        ret : (B, 1) risk score en [0,1]
        """
        out, _ = self.lstm(seq)          # (B, T, hidden)
        return self.sig(self.fc(out[:, -1, :]))   # último timestep

# ── Verificar con secuencia simulada desde batch real ──
print("Verificando LSTM...", end=" ")
lstm = RiskLSTM().to(DEVICE)

# Construir secuencia T=10 desde el batch de prueba
with torch.no_grad():
    embeds = []
    for bt in batch_grafos:
        bt = bt.to(DEVICE)
        emb = gcn(bt.x, bt.edge_index, bt.edge_index_g, bt.batch)
        embeds.append(emb)
    seq  = torch.stack(embeds, dim=1)   # (B, T, GCN_OUT)
    pred = lstm(seq)                    # (B, 1)

print("OK\n")
print(f"  Parámetros LSTM : {sum(p.numel() for p in lstm.parameters()):,}")
print(f"  Seq shape       : {seq.shape}   ← debe ser (4, 10, 128)")
print(f"  Output shape    : {pred.shape}  ← debe ser (4, 1)")
print(f"  Predicciones    : {pred.squeeze().tolist()}")
print(f"  Rango válido    : todas entre 0 y 1 → "
      f"{'✓' if pred.min() >= 0 and pred.max() <= 1 else '✗'}")

Verificando LSTM... OK

  Parámetros LSTM : 921,857
  Seq shape       : torch.Size([4, 10, 128])   ← debe ser (4, 10, 128)
  Output shape    : torch.Size([4, 1])  ← debe ser (4, 1)
  Predicciones    : [0.49070602655410767, 0.5075731873512268, 0.4983932375907898, 0.500287652015686]
  Rango válido    : todas entre 0 y 1 → ✓


In [11]:
# ─── Celda 09: Entrenamiento ──────────────────────────────────────────────────
EPOCHS      = 30
LR          = 1e-4
WD          = 1e-2
CLIP        = 0.5
ACCUM_STEPS = 4
PATIENCE    = 5

optimizer = AdamW(
    list(gcn.parameters()) + list(lstm.parameters()),
    lr=LR, weight_decay=WD)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

best_val      = float("inf")
patience_cnt  = 0
history       = {"train": [], "val": []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    gcn.train(); lstm.train()
    train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(dl_train, desc=f"Época {epoch:02d} train", leave=False)
    for step, (batched_t, risks) in enumerate(pbar):
        risks = risks.to(DEVICE)

        embeds = []
        for bt in batched_t:
            bt  = bt.to(DEVICE)
            emb = gcn(bt.x, bt.edge_index, bt.edge_index_g, bt.batch)
            embeds.append(emb)

        seq  = torch.stack(embeds, dim=1)   # (B, T, GCN_OUT)
        pred = lstm(seq)                    # (B, 1)
        loss = criterion(pred, risks) / ACCUM_STEPS
        loss.backward()
        train_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                list(gcn.parameters()) + list(lstm.parameters()), CLIP)
            optimizer.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item() * ACCUM_STEPS:.4f}")

    train_loss /= len(dl_train)

    # ── Validación ──
    gcn.eval(); lstm.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batched_t, risks in tqdm(dl_val, desc=f"Época {epoch:02d} val  ", leave=False):
            risks  = risks.to(DEVICE)
            embeds = []
            for bt in batched_t:
                bt  = bt.to(DEVICE)
                emb = gcn(bt.x, bt.edge_index, bt.edge_index_g, bt.batch)
                embeds.append(emb)
            seq      = torch.stack(embeds, dim=1)
            pred     = lstm(seq)
            val_loss += criterion(pred, risks).item()

    val_loss /= len(dl_val)
    scheduler.step()

    history["train"].append(train_loss)
    history["val"].append(val_loss)

    print(f"Época {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f}", end="")

    # ── Checkpoint ──
    torch.save({
        "epoch"    : epoch,
        "gcn"      : gcn.state_dict(),
        "lstm"     : lstm.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_loss" : val_loss
    }, CKPT_DIR / f"ckpt_ep{epoch:02d}.pt")

    # ── Early stopping ──
    if val_loss < best_val:
        best_val     = val_loss
        patience_cnt = 0
        torch.save(gcn.state_dict(),  CKPT_DIR / "gcn_best.pt")
        torch.save(lstm.state_dict(), CKPT_DIR / "lstm_best.pt")
        print(" ✓ mejor modelo")
    else:
        patience_cnt += 1
        print(f" (patience {patience_cnt}/{PATIENCE})")
        if patience_cnt >= PATIENCE:
            print(f"\nEarly stopping en época {epoch}")
            break

print(f"\nEntrenamiento completo | mejor val_loss={best_val:.4f}")

Época 01 | train=0.0067 | val=0.0414 ✓ mejor modelo


Época 02 | train=0.0029 | val=0.0296 ✓ mejor modelo


Época 03 | train=0.0027 | val=0.0417 (patience 1/5)


Época 04 | train=0.0024 | val=0.0381 (patience 2/5)


Época 05 | train=0.0024 | val=0.0382 (patience 3/5)


Época 06 | train=0.0023 | val=0.0413 (patience 4/5)


Época 07 | train=0.0022 | val=0.0416 (patience 5/5)

Early stopping en época 7

Entrenamiento completo | mejor val_loss=0.0296


In [ ]:
# ─── Celda 09b: Gráficas de entrenamiento ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Si history tiene datos del entrenamiento actual, úsalos;
# en caso contrario, usa los valores registrados en los logs.
_train = history["train"] if history["train"] else \n    [0.0067, 0.0029, 0.0027, 0.0024, 0.0024, 0.0023, 0.0022]
_val   = history["val"]   if history["val"]   else \n    [0.0414, 0.0296, 0.0417, 0.0381, 0.0382, 0.0413, 0.0416]

epochs  = list(range(1, len(_train) + 1))
best_ep = int(epochs[_val.index(min(_val))])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("GCN-LSTM · Curvas de entrenamiento", fontsize=14, fontweight="bold")

# ── Izquierda: pérdida absoluta ──
ax = axes[0]
ax.plot(epochs, _train, "o-",  color="#2196F3", lw=2, label="Train loss")
ax.plot(epochs, _val,   "s--", color="#F44336", lw=2, label="Val loss")
ax.axvline(best_ep, color="green", ls=":", lw=1.5,
           label=f"Mejor época ({best_ep})")
ax.scatter([best_ep], [min(_val)], color="green", s=90, zorder=5)
ax.set_xlabel("Época", fontsize=11)
ax.set_ylabel("MSE Loss", fontsize=11)
ax.set_title("Pérdida MSE por época")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
ax.grid(True, alpha=0.35)

# ── Derecha: escala log ──
ax2 = axes[1]
ax2.semilogy(epochs, _train, "o-",  color="#2196F3", lw=2, label="Train loss")
ax2.semilogy(epochs, _val,   "s--", color="#F44336", lw=2, label="Val loss")
ax2.axvline(best_ep, color="green", ls=":", lw=1.5,
            label=f"Mejor época ({best_ep})")
ax2.set_xlabel("Época", fontsize=11)
ax2.set_ylabel("MSE Loss (log)", fontsize=11)
ax2.set_title("Pérdida MSE — escala logarítmica")
ax2.legend(fontsize=10)
ax2.grid(True, which="both", alpha=0.35)

plt.tight_layout()
plt.savefig("training_curves_gcn_lstm.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mejor val_loss={min(_val):.4f} en época {best_ep}")
print(f"Early stopping activado en época {len(epochs)}")


In [1]:
# ─── Celda 10: Evaluación ─────────────────────────────────────────────────────
from scipy.stats import pearsonr

# Cargar mejores pesos
gcn.load_state_dict(torch.load(CKPT_DIR / "gcn_best.pt",
                                map_location=DEVICE, weights_only=True))
lstm.load_state_dict(torch.load(CKPT_DIR / "lstm_best.pt",
                                 map_location=DEVICE, weights_only=True))
gcn.eval(); lstm.eval()

all_pred, all_true = [], []

with torch.no_grad():
    for batched_t, risks in tqdm(dl_val, desc="Evaluando val"):
        risks  = risks.to(DEVICE)
        embeds = []
        for bt in batched_t:
            bt  = bt.to(DEVICE)
            emb = gcn(bt.x, bt.edge_index, bt.edge_index_g, bt.batch)
            embeds.append(emb)
        seq  = torch.stack(embeds, dim=1)
        pred = lstm(seq)
        all_pred.extend(pred.cpu().squeeze().tolist())
        all_true.extend(risks.cpu().squeeze().tolist())

all_pred = np.array(all_pred)
all_true = np.array(all_true)

mae        = np.mean(np.abs(all_pred - all_true))
mse        = np.mean((all_pred - all_true)**2)
corr, pval = pearsonr(all_pred, all_true)

print(f"── Métricas en validación ──")
print(f"  MAE       : {mae:.4f}")
print(f"  MSE       : {mse:.4f}")
print(f"  Pearson r : {corr:.4f}  (p={pval:.4f})")
print(f"\n  Predicciones — min={all_pred.min():.4f} | max={all_pred.max():.4f} | media={all_pred.mean():.4f}")
print(f"  Labels    — min={all_true.min():.4f} | max={all_true.max():.4f} | media={all_true.mean():.4f}")

NameError: name 'gcn' is not defined

In [ ]:
# ─── Celda 10b: Gráficas de evaluación ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("GCN-LSTM · Evaluación en validación", fontsize=14, fontweight="bold")

# ── Scatter: pred vs true ──
ax = axes[0]
ax.scatter(all_true, all_pred, alpha=0.4, s=15, color="#673AB7", edgecolors="none")
lim = [min(all_true.min(), all_pred.min()) - 0.02,
       max(all_true.max(), all_pred.max()) + 0.02]
ax.plot(lim, lim, "r--", lw=1.5, label="Predicción perfecta")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Risk score real",       fontsize=11)
ax.set_ylabel("Risk score predicho",   fontsize=11)
ax.set_title(f"Scatter  (r={corr:.3f})")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Histograma del error ──
ax2 = axes[1]
errors = all_pred - all_true
ax2.hist(errors, bins=40, color="#009688", edgecolor="white", alpha=0.85)
ax2.axvline(0, color="red", ls="--", lw=1.5)
ax2.set_xlabel("Error (pred − real)", fontsize=11)
ax2.set_ylabel("Frecuencia",           fontsize=11)
ax2.set_title(f"Distribución del error  (MAE={mae:.4f})")
ax2.grid(True, alpha=0.3)

# ── Histograma: pred vs true superpuestos ──
ax3 = axes[2]
ax3.hist(all_true, bins=40, alpha=0.55, color="#F44336", label="Real")
ax3.hist(all_pred, bins=40, alpha=0.55, color="#2196F3", label="Predicho")
ax3.set_xlabel("Risk score", fontsize=11)
ax3.set_ylabel("Frecuencia",  fontsize=11)
ax3.set_title("Distribución de scores")
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("eval_plots_gcn_lstm.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"MSE={mse:.4f} | MAE={mae:.4f} | Pearson r={corr:.4f} (p={pval:.4f})")


In [13]:
# ─── Celda 11: Guardar embeddings y risk scores ───────────────────────────────
gcn.eval(); lstm.eval()

all_embeds, all_scores, all_meta = [], [], []

for split_name, split_ds, split_dl in [
    ("train", ds_train, dl_train),
    ("val",   ds_val,   dl_val),
    ("test",  ds_test,  dl_test)
]:
    idx_global = 0
    with torch.no_grad():
        for batched_t, risks in tqdm(split_dl, desc=f"Inferencia {split_name}"):
            embeds = []
            for bt in batched_t:
                bt  = bt.to(DEVICE)
                emb = gcn(bt.x, bt.edge_index, bt.edge_index_g, bt.batch)
                embeds.append(emb)

            seq        = torch.stack(embeds, dim=1)   # (B, T, 128)
            pred       = lstm(seq)                    # (B, 1)
            hidden_emb = seq[:, -1, :]                # (B, 128) embedding final

            all_embeds.append(hidden_emb.cpu().numpy())
            all_scores.append(pred.cpu().squeeze().numpy())

            B = risks.shape[0]
            for i in range(B):
                win = split_ds.windows[idx_global + i]
                all_meta.append({
                    "seq_id"    : win.iloc[0]["seq_id"],
                    "frames"    : "|".join(win["image_id"].tolist()),
                    "risk_score": round(float(pred[i].item()), 4),
                    "split"     : split_name
                })
            idx_global += B

# ── Concatenar y guardar ──
embeddings = np.concatenate(all_embeds, axis=0)
scores     = np.concatenate(all_scores, axis=0)
meta_df    = pd.DataFrame(all_meta)

np.save(OUT_DIR / "embeddings_spatiotemporal.npy", embeddings)
np.save(OUT_DIR / "risk_scores.npy", scores)
meta_df.to_csv(OUT_DIR / "scene_manifest.csv", index=False)

print(f"── Outputs guardados en {OUT_DIR} ──")
print(f"  embeddings_spatiotemporal.npy : {embeddings.shape}")
print(f"  risk_scores.npy               : {scores.shape}")
print(f"  scene_manifest.csv            : {len(meta_df):,} filas")
print(f"\n  Risk scores — min={scores.min():.4f} | max={scores.max():.4f} | media={scores.mean():.4f}")

Inferencia test: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 452/452 [01:32<00:00,  4.87it/s]


── Outputs guardados en /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/spatiotemporal_outputs ──
  embeddings_spatiotemporal.npy : (12175, 128)
  risk_scores.npy               : (12175,)
  scene_manifest.csv            : 12,175 filas

  Risk scores — min=0.0554 | max=0.8768 | media=0.7487
